# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nancy121205/flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

---
**Task type: Ranking (using a priority score).**

The goal is to rank content pages from highest to lowest review priority. A content editor can use the ranked list to decide which pages to investigate first.

The output is a review queue, not an automatic instruction to rewrite a page. A high rank means a page deserves investigation, not that its content is definitely poor.

A wrong ranking can waste an editor's time or leave a page needing attention too far down the queue. Results will be used for decision support only.

---


In [1]:
# Confirm the chosen task and the decision it supports.
task_type = "ranking"
decision = "Which content pages should an editor review first?"

assert task_type == "ranking"
print("Task:", task_type)
print("Decision:", decision)
print("Output: a prioritized review queue")

Task: ranking
Decision: Which content pages should an editor review first?
Output: a prioritized review queue


## 2. Target or proxy

**Ideal target:** The actual improvement in clicks a page could achieve after review. This outcome is not available directly in the starter dataset.

**Available proxy:** `is_declining_label`, defined as 1 when `trend_direction == "down"` and 0 otherwise.

This proxy identifies pages with declining impressions over the comparison window. It is a rule-derived label, not proof that a page has untapped click potential or that editing it will improve performance.

The ranking will prioritize pages associated with this proxy. A future evaluation should use an observed outcome measured after the review decision.


In [2]:
# The dataset will be loaded in Section 4.
# These checks document the proxy definition.
proxy_name = "is_declining_label"
proxy_source = "trend_direction"
proxy_rule = "down means 1; otherwise 0"

print("Proxy:", proxy_name)
print("Source:", proxy_source)
print("Rule:", proxy_rule)

# Leakage warning:
# Never use trend_direction or trend_pct as ranking features
# when the proxy is derived from trend_direction.

Proxy: is_declining_label
Source: trend_direction
Rule: down means 1; otherwise 0


## 3. Success metric

**Primary metric: Precision@K, with K = 20 and K = 50.**

Precision@K is the fraction of the top K ranked pages that have the decline proxy label.

For example, Precision@20 of 0.60 would mean that 12 of the first 20 pages are flagged as declining. This is an illustration, not a measured result.

The baseline is the overall proportion of pages carrying the proxy label. The ranking should achieve Precision@20 and Precision@50 above this baseline on held-out clients before it can be considered useful.

These metrics evaluate agreement with the decline proxy, not the actual benefit of editing a page.


In [6]:

# Section 3: Success metric
# Run the Section 4 data-loading cell first.

K_VALUES = (20, 50)

if "df" not in globals():
    print("Please run the Section 4 data-loading cell first.")
else:
    if "is_declining_label" not in df.columns:
        raise ValueError(
            "Missing is_declining_label. Run the Section 4 "
            "cell that creates the proxy label."
        )

    base_rate = df["is_declining_label"].mean()

    print(f"Proxy-label baseline rate: {base_rate:.3f}")

    for k in K_VALUES:
        if len(df) >= k:
            print(f"Evaluation metric: Precision@{k}")
        else:
            print(
                f"Precision@{k}: dataset has fewer than {k} rows"
            )

    print("Success criterion: beat the baseline on held-out clients.")

Proxy-label baseline rate: 0.542
Evaluation metric: Precision@20
Evaluation metric: Precision@50
Success criterion: beat the baseline on held-out clients.


## 4. The unit of analysis, as a real dataframe

**Unit of analysis: one pseudonymized content page.**

The starter dataset contains a snapshot of content metadata and aggregated performance metrics. Each row represents one content item, rather than one client or one daily event.

I will inspect the actual columns, confirm the row grain, and construct the proxy label from the provided trend direction. I will exclude identifiers and label-derived columns from future ranking features.


In [5]:
from pathlib import Path
import pandas as pd

# Prefer the local repository copy. Fall back to the public
# anonymized starter dataset if the file is not present locally.
possible_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]

csv_path = next((p for p in possible_paths if p.exists()), None)

if csv_path is not None:
    df = pd.read_csv(csv_path)
    print("Loaded local dataset:", csv_path)
else:
    url = (
        "https://raw.githubusercontent.com/"
        "flyrank-bih/flyrank-ml-internship-starter/"
        "main/data/raw/content_refresh_anonymized.csv"
    )
    df = pd.read_csv(url)
    print("Loaded the public anonymized starter dataset.")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# Check the expected unit of analysis.
assert "content_id" in df.columns, "Missing content_id"
assert df["content_id"].is_unique, (
    "Expected one row per content item; investigate duplicate IDs."
)
assert "trend_direction" in df.columns, (
    "Cannot construct the decline proxy without trend_direction."
)

# Construct the documented, rule-derived proxy.
df["is_declining_label"] = (
    df["trend_direction"].eq("down").astype("int8")
)

print("\nProxy-label distribution:")
print(df["is_declining_label"].value_counts(dropna=False))
print("\nProxy-label rate:",
      round(df["is_declining_label"].mean(), 4))

# Show a small, privacy-safe sample.
display_cols = [
    c for c in [
        "content_id",
        "content_type",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "trend_direction",
        "is_declining_label",
    ]
    if c in df.columns
]

display(df[display_cols].head())

Loaded the public anonymized starter dataset.
Shape: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Proxy-label distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Proxy-label rate: 0.5421


,content_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,trend_direction,is_declining_label
0,content_304f48230142,keyword article,3803,29,0.76,10.6,down,1
1,content_a1fb4e703a9e,keyword article,15320,7,0.05,20.3,down,1
2,content_9aa793d4d895,keyword article,12581,11,0.09,36.5,down,1
3,content_331d6c4de07b,keyword article,11751,58,0.49,6.2,stable,0
4,content_d99b7a2d90ca,keyword article,19140,24,0.13,44.0,down,1


## 5. Why ML beats a fixed rule here

A fixed rule can prioritize pages using one threshold, such as low CTR or declining impressions. However, review priority may depend on several factors together, including impressions, clicks, average position, content age, update history, and keyword context.

A ranking model could learn combinations of these signals and produce a more useful review queue than a single threshold. However, that advantage has not yet been demonstrated. A transparent fixed-rule ranking is an important baseline, and the model must outperform it on held-out clients before we claim a benefit.

The starter dataset is a single aggregated snapshot, and its decline proxy is rule-derived. Therefore, any results at this stage are exploratory and support prioritization decisions; they do not establish that ML improves content or causes more clicks.


In [7]:
# Inspect whether multiple candidate signals vary in the dataset.
# This supports investigation; it does NOT prove ML will win.

candidate_signals = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "search_volume",
    "word_count",
]

available_signals = [
    c for c in candidate_signals if c in df.columns
]

print("Available candidate signals:", available_signals)

display(
    df[available_signals]
    .describe()
    .T
    .round(2)
)

print("\nDistinct values per signal:")
for col in available_signals:
    print(f"{col}: {df[col].nunique(dropna=True)}")

print(
    "\nInterpretation: varied signals provide candidates for testing "
    "whether a multivariable ranking helps. They do not prove it."
)

# Keep identifiers and proxy-derived fields out of model features.
excluded_from_features = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
}

candidate_features = [
    c for c in available_signals
    if c not in excluded_from_features
]

print("\nPotential numeric features for later experiments:")
print(candidate_features)

Available candidate signals: ['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'content_age_days', 'days_since_last_update', 'search_volume', 'word_count']


,count,mean,std,min,25%,50%,75%,max
impressions_90d,30000.0,5200.37,16838.02,1.0,81.0,731.00,3615.25,517715.0
clicks_90d,30000.0,16.10,75.08,0.0,0.0,1.00,7.00,4178.0
ctr,30000.0,0.51,3.28,0.0,0.0,0.07,0.29,100.0
avg_position,30000.0,16.34,15.22,0.0,6.2,10.80,22.30,245.0
content_age_days,30000.0,256.17,132.71,90.0,132.0,236.00,333.00,564.0
days_since_last_update,30000.0,46.10,42.08,1.0,20.0,20.00,104.00,373.0
search_volume,27532.0,158.88,1518.27,0.0,0.0,10.00,20.00,74000.0
word_count,22301.0,3107.76,1452.38,8.0,2413.0,2877.00,3666.00,9546.0



Distinct values per signal:
impressions_90d: 9438
clicks_90d: 477
ctr: 401
avg_position: 869
content_age_days: 225
days_since_last_update: 57
search_volume: 41
word_count: 5476

Interpretation: varied signals provide candidates for testing whether a multivariable ranking helps. They do not prove it.

Potential numeric features for later experiments:
['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'content_age_days', 'days_since_last_update', 'search_volume', 'word_count']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.